# 01허깅페이스에서_모델받아_다국어번역_서비스만들기

In [3]:
!pip install transformers

In [6]:
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer

ko_text = "이것은 m2m모델로 만든 다국어 번역기 입니다."
chinese_text = "生活就像一盒巧克力。"

model = M2M100ForConditionalGeneration.from_pretrained("facebook/m2m100_1.2B")
tokenizer = M2M100Tokenizer.from_pretrained("facebook/m2m100_1.2B")

# translate KOREAN to ENGLISH
tokenizer.src_lang = "hi"
encoded_hi = tokenizer(ko_text, return_tensors="pt")
generated_tokens = model.generate(**encoded_hi, forced_bos_token_id=tokenizer.get_lang_id("en"))  # 바꿀 언어
result1 = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
print(result1)
# => "La vie est comme une boîte de chocolat."

# translate KOREAN to JAPANESE
tokenizer.src_lang = "ko"
encoded_zh = tokenizer(ko_text, return_tensors="pt")
generated_tokens = model.generate(**encoded_zh, forced_bos_token_id=tokenizer.get_lang_id("ja"))
result2 = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
result2
# => "Life is like a box of chocolate."


['This is a multi-language translator made with the m2m model.']


['これは、m2mモデルで作成された多言語翻訳機です。']

In [8]:
import gradio as gr
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer
import torch

# 모델 및 토크나이저 로드
model_name = "facebook/m2m100_1.2B"
tokenizer = M2M100Tokenizer.from_pretrained(model_name)
model = M2M100ForConditionalGeneration.from_pretrained(model_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# 지원 언어 목록 (원하는 언어 코드를 추가하세요)
languages = {
    "English": "en",
    "Korean": "ko",
    "Japanese": "ja",
    "Chinese": "zh",
    "Hindi": "hi"
}

def translate_fn(text, target_lang):
    # 소스 언어 고정: 한국어("ko")
    tokenizer.src_lang = "ko"
    inputs = tokenizer(text, return_tensors="pt").to(device)
    generated = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.get_lang_id(languages[target_lang])
    )
    return tokenizer.batch_decode(generated, skip_special_tokens=True)[0]

# CSS로 오른쪽 컬럼 하단 정렬
css = """
.lang-col {
    display: flex;
    flex-direction: column;
    justify-content: flex-end;
    height: 100%;
}
"""

with gr.Blocks(css=css) as demo:
    gr.Markdown("### 다국어 번역기 (M2M100 기반)")
    with gr.Row():
        with gr.Column(scale=3):
            input_text = gr.Textbox(label="번역할 텍스트", lines=6, placeholder="여기에 텍스트 입력")
            output_text = gr.Textbox(label="번역 결과", lines=6)
        with gr.Column(scale=1, elem_classes="lang-col"):
            lang = gr.Dropdown(
                choices=list(languages.keys()),
                label="번역할 언어 선택"
            )
    translate_btn = gr.Button("번역하기")
    translate_btn.click(fn=translate_fn, inputs=[input_text, lang], outputs=output_text)

demo.launch()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [9]:
demo.close()

Closing server running on port: 7860


In [11]:
!pip install datasets soundfile

   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 12.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/25.8 MB ? eta -:--:--
   ---- ----------------------------------- 2.9/25.8 MB 15.2 MB/s eta 0:00:02
   --------- ------------------------------ 6.0/25.8 MB 14.2 MB/s eta 0:00:02
   -------------- ------------------------- 9.4/25.8 MB 14.7 MB/s eta 0:00:02
   ------------------- -------------------- 12.6/25.8 MB 14.9 MB/s eta 0:00:01
   ------------------------ --------------- 16.0/25.8 MB 15.0 MB/s eta 0:00:01
   ------------------------------ --------- 19.4/25.8 MB 15.1 MB/s eta 0:00:01
   ---------------------------------- ----- 22.3/25.8 MB 15.2 MB/s eta 0:00:01
   ---------------------------------------  25.7/25.8 MB 15.1 MB/s eta 0:00:01
   ---------------------------------------- 25.8/25.8 MB 13.5 MB/s eta 0:00:00

   -- -------------------------------------  1/16 [pycparser]
   -- -------

In [12]:
import gradio as gr
import torch
import numpy as np
from transformers import (
    M2M100ForConditionalGeneration, M2M100Tokenizer,
    SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan
)
from datasets import load_dataset

# ── 1) 번역 모델 로드 ─────────────────────────────────────────
mt_model_name = "facebook/m2m100_1.2B"
mt_tokenizer  = M2M100Tokenizer.from_pretrained(mt_model_name)
mt_model      = M2M100ForConditionalGeneration.from_pretrained(mt_model_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
mt_model.to(device)

# ── 2) TTS 모델 로드 ─────────────────────────────────────────
tts_processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")
tts_model     = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts").to(device)
tts_vocoder   = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan").to(device)

# 미리 한 명의 speaker x-vector 로드 (예시: 첫 번째 검증셋 화자)
embds = load_dataset("Matthijs/cmu-arctic-xvectors", split="validation")
speaker_embedding = torch.tensor(embds[0]["xvector"]).unsqueeze(0).to(device)

# ── 3) 지원 언어 맵핑 ─────────────────────────────────────────
languages = {
    "English": "en",
    "Korean":  "ko",
    "Japanese":"ja",
    "Chinese": "zh",
    "Hindi":   "hi",
}

# ── 4) 번역 + (영어일 때) TTS 합성 함수 ────────────────────────
def translate_and_tts(text, target_lang_name):
    # 1) 번역
    mt_tokenizer.src_lang = "ko"  # 입력은 한국어 고정
    inputs = mt_tokenizer(text, return_tensors="pt").to(device)
    gen = mt_model.generate(
        **inputs,
        forced_bos_token_id=mt_tokenizer.get_lang_id(languages[target_lang_name])
    )
    translated = mt_tokenizer.batch_decode(gen, skip_special_tokens=True)[0]
    
    # 2) 영어 번역일 경우 음성 합성
    if languages[target_lang_name] == "en":
        tts_inputs = tts_processor(text=translated, return_tensors="pt").to(device)
        speech = tts_model.generate_speech(
            tts_inputs["input_ids"],
            speaker_embedding,
            vocoder=tts_vocoder
        )
        # Gradio Audio는 (numpy_array, 샘플레이트) 형태를 허용
        return translated, (speech.cpu().numpy(), 16000)
    else:
        return translated, None

# ── 5) Gradio UI 구성 ─────────────────────────────────────────
css = """
.lang-col {
    display: flex;
    flex-direction: column;
    justify-content: flex-end;
    height: 100%;
}
"""

with gr.Blocks(css=css) as demo:
    gr.Markdown("### 다국어 번역기 + 영어 TTS 읽어주기")
    with gr.Row():
        with gr.Column(scale=3):
            inp = gr.Textbox(label="번역할 텍스트", lines=5, placeholder="여기에 한국어 입력")
            out_txt = gr.Textbox(label="번역 결과", lines=5)
            out_audio = gr.Audio(label="음성 출력 (영어 번역 시)", type="numpy")
        with gr.Column(scale=1, elem_classes="lang-col"):
            lang = gr.Dropdown(choices=list(languages.keys()), value="English", label="번역 언어 선택")
    btn = gr.Button("번역 + 재생")
    btn.click(
        fn=translate_and_tts,
        inputs=[inp, lang],
        outputs=[out_txt, out_audio]
    )

demo.launch()


C:\Users\Admin\miniforge3\envs\ai_serving\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--microsoft--speecht5_tts. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\Admin\miniforge3\envs\ai_serving\Lib\site-packages\huggingface_hub\file_download.py:143: UserWa

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
